# FAseg — Batch Inference on In-Vivo Data (1 run)

Runs inference for the **Full pretraining × best_s3 fine-tuned model** on
in-vivo data (`syf_upper_leg`).  Click "Run All".

Each run saves to `outputs/inference_results/invivo/<exp>_<strategy>/`:
- `pred_mask_3d.npy` / `.bin` — 3-D binary mask
- `boundary_matrix.npy` — TOF boundary per row/slice
- `inference_time.txt`


In [1]:
import os, sys, json, time
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

_NB_DIR = os.path.abspath('')
if os.path.basename(_NB_DIR) == 'scripts':
    _project_root = os.path.dirname(_NB_DIR)
else:
    _project_root = _NB_DIR
_src_root = os.path.join(_project_root, 'src')
if _src_root not in sys.path:
    sys.path.insert(0, _src_root)

from faseg.models import UNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


In [2]:
# =========================================================================
# Config — fine-tuning checkpoint to run inference on
# =========================================================================
BIN_PATH = os.path.join(_project_root, 'manual segmentation', 'syf_upper_leg')

# (pretrain_dir, ft_strategy) — only the Full pretraining, best_s3 fine-tuned model
FT_RUNS = [
    ('outputs/pretraining',                'finetuning_best_s3'),
]

print(f'Will run inference for {len(FT_RUNS)} fine-tuned models')


Will run inference for 1 fine-tuned models


In [3]:
# =========================================================================
# Load & preprocess the .bin file once (same for all models)
# =========================================================================
def load_and_preprocess(bin_path):
    raw = np.fromfile(bin_path, dtype=np.float32)
    S = raw.size // (512 * 876)
    data = raw.reshape((S, 512, 876))

    # Circular shift per slice (TOF acquisition correction)
    for i in range(S):
        shift = i if S == 512 else (2 * i)
        data[i] = np.concatenate([data[i, shift:], data[i, :shift]], axis=0)

    # Crop
    sub = data[:, 64:448, 300:684].astype(np.float32)

    # Per-slice normalise + clip
    for i in range(S):
        vmax = np.abs(sub[i]).max()
        if vmax > 0:
            sub[i] /= vmax
    sub = np.clip(sub, -1.0, 1.0)
    return sub

print('Loading & preprocessing in-vivo data ...')
stack = load_and_preprocess(BIN_PATH)
S, H, W = stack.shape
print(f'Preprocessed: {stack.shape}  range=[{stack.min():.3f}, {stack.max():.3f}]')

# DataLoader (reused for all models)
tensor_stack = torch.from_numpy(stack).unsqueeze(1)
dataset = TensorDataset(tensor_stack)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=4)

Loading & preprocessing in-vivo data ...
Preprocessed: (512, 384, 384)  range=[-1.000, 1.000]


In [4]:
# =========================================================================
# Run inference for one fine-tuned model
# =========================================================================
def run_inference_for_model(save_dir, model, loader, S, H):
    """Run inference and save results to *save_dir*."""
    os.makedirs(save_dir, exist_ok=True)

    all_masks = []
    t0 = time.time()
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(device)
            logits = model(batch)
            preds = logits.argmax(dim=1).cpu().numpy().astype(np.uint8)
            all_masks.append(preds)
    pred_masks = np.concatenate(all_masks, axis=0)
    elapsed = time.time() - t0

    # Boundary extraction
    boundary_matrix = np.full((H, S), np.nan, dtype=np.float32)
    for s in range(S):
        for row in range(H):
            ones = np.where(pred_masks[s, row, :] == 1)[0]
            if len(ones) > 0:
                boundary_matrix[row, s] = float(ones[0])

    # Save
    mask_3d = pred_masks.transpose(1, 2, 0)
    np.save(os.path.join(save_dir, 'pred_mask_3d.npy'), mask_3d)
    np.asfortranarray(mask_3d).tofile(os.path.join(save_dir, 'pred_mask_3d.bin'))
    np.save(os.path.join(save_dir, 'boundary_matrix.npy'), boundary_matrix)
    with open(os.path.join(save_dir, 'inference_time.txt'), 'w') as f:
        f.write(f'{elapsed:.2f}s  ({S} slices, {S/elapsed:.0f} slices/s)\\n')

    return elapsed

In [5]:
# =========================================================================
# Run all
# =========================================================================
results = []
t_start = time.time()
INFERENCE_ROOT = os.path.join(_project_root, 'outputs', 'inference_results', 'invivo')

for pretrain_rel, ft_strategy in FT_RUNS:
    pretrain_dir = os.path.join(_project_root, pretrain_rel)
    ft_dir = os.path.join(pretrain_dir, ft_strategy)
    ckpt_path = os.path.join(ft_dir, 'finetune_best.pth')

    # Subfolder name:  e.g. "Full_latest", "No_TOF_best_s3"
    exp_name = os.path.basename(pretrain_dir)         # "pretraining", "pretraining_no_tof", ...
    strategy_short = ft_strategy.replace('finetuning_', '')   # "latest", "best_s3"
    label = f'{exp_name}_{strategy_short}'             # "pretraining_latest", ...

    save_dir = os.path.join(INFERENCE_ROOT, label)
    print(f'\n{"="*60}')
    print(f'  {label}')
    print(f'  Save to: {save_dir}')
    print(f'{"="*60}')

    if not os.path.exists(ckpt_path):
        print(f'  !! SKIP: {ckpt_path} not found')
        results.append((label, 'SKIP'))
        continue

    # Read model config
    config_path = os.path.join(pretrain_dir, 'config.txt')
    if os.path.exists(config_path):
        with open(config_path) as f:
            cfg = json.load(f)
        base_ch = cfg.get('base_channel', 64)
        dropout = cfg.get('dropout_prob', 0.2)
        use_bn  = cfg.get('use_bn', True)
    else:
        base_ch, dropout, use_bn = 64, 0.2, True

    # Build & load model
    model = UNet(in_ch=1, base_ch=base_ch, num_classes=2,
                 dropout_prob=dropout, use_bn=use_bn).to(device)
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state)
    model.eval()

    # Run
    elapsed = run_inference_for_model(save_dir, model, loader, S, H)
    print(f'  DONE — {S} slices in {elapsed:.1f}s  ({S/elapsed:.0f} slices/s)')
    results.append((label, f'{elapsed:.1f}s'))

# =========================================================================
# Summary
# =========================================================================
print(f'\n{"="*60}')
print(f'  SUMMARY  (total: {(time.time()-t_start)/60:.0f} min)')
print(f'{"="*60}')
for label, status in results:
    print(f'  {label:45s}  {status}')


  pretraining_best_s3
  Save to: /data/projects/AgentWork/FAseg for github/outputs/inference_results/invivo/pretraining_best_s3


/home/yifei-sun/anaconda3/envs/faseg_ablation/lib/python3.10/site-packages/torch/cuda/__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


  DONE — 512 slices in 2.7s  (190 slices/s)

  SUMMARY  (total: 0 min)
  pretraining_best_s3                            2.7s
